|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>Why you cannot decode tokens one at a time<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
print('vocab', tokenizer.vocab_size)

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab 151643


# You cannot decode tokens one at a time

Streaming means emitting text as each token arrives. The obvious
implementation is `decode(token)` per token, concatenated.

It works for most text, which is why it ships.

In [2]:
def naive_stream(token_ids):
  """Decode each token alone, and join the pieces."""
  return ''.join(tokenizer.decode([token_id]) for token_id in token_ids)

for text in ['Hello world, how are you?', 'print("hi")  # done']:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  print(f'{text!r}')
  print(f'  naive == batch: {naive_stream(token_ids) == tokenizer.decode(token_ids)}')

'Hello world, how are you?'
  naive == batch: True
'print("hi")  # done'
  naive == batch: True


### And then it does not

In [3]:
broken = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466 family',
          '\U0001F3F3\ufe0f\u200d\U0001F308 pride']
for text in broken:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  naive, correct = naive_stream(token_ids), tokenizer.decode(token_ids)
  print(f'pieces : {tokenizer.convert_ids_to_tokens(token_ids)}')
  print(f'naive  : {naive!r}')
  print(f'correct: {correct!r}')
  print(f'match  : {naive == correct}\n')

pieces : ['ðŁĳ¨', 'âĢ', 'į', 'ðŁĳ©', 'âĢ', 'į', 'ðŁĳ§', 'âĢ', 'į', 'ðŁĳ¦', 'Ġfamily']
naive  : '👨��👩��👧��👦 family'
correct: '👨\u200d👩\u200d👧\u200d👦 family'
match  : False

pieces : ['ðŁı³', 'ï¸ı', 'âĢ', 'į', 'ðŁĮĪ', 'Ġpride']
naive  : '🏳️��🌈 pride'
correct: '🏳️\u200d🌈 pride'
match  : False



Look at the pieces. The zero-width joiner that glues the family emoji
together is three bytes, and the tokenizer split it across two tokens. Decode
either of them alone and you get a replacement character, because half a
character is not a character.

The same thing happens to accented Latin, to CJK, and to all other text whose
UTF-8 encoding is longer than one byte. It happens too rarely for your tests
to catch it, and often enough that users report mojibake.

# Decode the prefix, emit the difference

Keep the token ids. Decode all of them. Emit whatever is longer than what you
emitted last time.

The tokenizer sees whole characters again, because it is looking at whole
sequences.

In [4]:
class IncrementalDetokenizer:
  """Turn a stream of token ids into a stream of complete characters."""
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.token_ids = []
    self.num_emitted = 0      # the characters that the client has already

  def push(self, token_id):
    """-> the NEW text that this token completes. It can be ''."""
    self.token_ids.append(token_id)
    text = self.tokenizer.decode(self.token_ids)
    # A replacement character at the end means that the last character is
    # not complete. Hold it back. The next token completes it.
    if text.endswith('\ufffd'):
      return ''
    return self._emit_from(text)

  def flush(self):
    """The stream ended. Send the rest, complete or not."""
    return self._emit_from(self.tokenizer.decode(self.token_ids))

  def _emit_from(self, text):
    new_text = text[self.num_emitted:]
    self.num_emitted = len(text)
    return new_text

def stream_text(tokenizer, token_ids):
  """Send token_ids through a new detokenizer. -> all the text that it sent."""
  detokenizer = IncrementalDetokenizer(tokenizer)
  pushed = ''.join(detokenizer.push(token_id) for token_id in token_ids)
  return pushed + detokenizer.flush()

for text in broken + ['caf\u00e9 na\u00efve \u65e5\u672c\u8a9e']:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  streamed = stream_text(tokenizer, token_ids)
  print(f'{streamed == tokenizer.decode(token_ids)}  {streamed!r}')

True  '👨\u200d👩\u200d👧\u200d👦 family'
True  '🏳️\u200d🌈 pride'
True  'café naïve 日本語'


### Fuzz it, because examples prove nothing

A streaming detokenizer has one job and it is an equality: whatever the
stream emits, concatenated, must equal what a batch decode would have
produced. Test that on a lot of text rather than on four strings you thought
of.

In [5]:
import random
rng = random.Random(0)
failures = 0
for trial in range(300):
  token_ids = [rng.randrange(tokenizer.vocab_size) for _ in range(rng.randint(1, 40))]
  if stream_text(tokenizer, token_ids) != tokenizer.decode(token_ids):
    failures += 1
    if failures == 1:
      print('first failure:', tokenizer.convert_ids_to_tokens(token_ids)[:10])
print(f'{failures} failures in 300 random sequences')

0 failures in 300 random sequences


# The other half: stop strings

A request says "stop when you see `END`". The model emits ` EN` and then `D`.
Neither token contains the stop string. The two tokens together contain it.

So you cannot examine tokens. You must examine the **text**, over a window
that crosses the token boundaries. You must also hold back all text that can
become the start of a stop string.

In [6]:
def held_length(held, stop):
  """The length of the longest end of `held` that is a start of `stop`.
  That text can still become the stop string, so do not emit it yet."""
  for length in range(min(len(stop) - 1, len(held)), 0, -1):
    if held.endswith(stop[:length]):
      return length
  return 0

def stream_until_stop(tokenizer, token_ids, stop):
  """Emit text until `stop` appears. Never emit a start of `stop`.
  -> (the emitted text, True if the stop string appeared).

  The stop string can cross a token boundary, so you cannot look at tokens.
  And you cannot emit early: 'EN' must not show before you know that it
  was the start of 'END'.
  """
  detokenizer = IncrementalDetokenizer(tokenizer)
  emitted, held = [], ''
  for token_id in token_ids:
    held += detokenizer.push(token_id)
    stop_at = held.find(stop)
    if stop_at >= 0:
      emitted.append(held[:stop_at])
      return ''.join(emitted), True
    keep = held_length(held, stop)
    emitted.append(held[:len(held) - keep])
    held = held[len(held) - keep:]
  return ''.join(emitted) + held + detokenizer.flush(), False

for text in ['Answer: yes. END OF LINE', 'no stop here at all']:
  token_ids = tokenizer(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(tokenizer, token_ids, 'END')
  print(f'{text!r}')
  print(f'  emitted {emitted!r}, stopped={stopped}\n')

'Answer: yes. END OF LINE'
  emitted 'Answer: yes. ', stopped=True

'no stop here at all'
  emitted 'no stop here at all', stopped=False



The text that you hold back makes this difficult. If you emit too early, the
user sees `EN` on the screen. You then find that it is the start of the stop
string, and you cut the stream. If you hold back too much, the stream
stutters.

None of this is interesting. It causes most of the bugs that users see in a
real server. That is a good reason to do it correctly one time, and to cover
it with a fuzz test.

    ./vc guide 14